In [9]:
import pandas as pd

# Charger le dataset
df = pd.read_csv('Clean_Dataset.csv')
df

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
...,...,...,...,...,...,...,...,...,...,...,...,...
300148,300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265
300149,300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105
300150,300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099
300151,300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585


Test 1 :


Hypothèse : Le prix moyen des billets en classe Business est significativement plus élevé que celui de la classe Economy.

Test à utiliser : T-test (test de Student) pour échantillons indépendants.

Variables : class (indépendante) et price (dépendante).



In [8]:
from scipy.stats import ttest_ind, pearsonr, f_oneway
grpeconomy=df[df["class"]=="Economy"]["price"]
grpbusiness=df[df["class"]=="Business"]["price"]
t_stat, p_val_class = ttest_ind(grpeconomy, grpbusiness)
print(f"P-value Classe : {p_val_class:.5f}")

P-value Classe : 0.00000


La p-value < 5%, on rejette H0 (l'idée que les prix seraient les mêmes),donc ee prix moyen des billets en classe Business est significativement plus élevé que celui de la classe Economy


Test 2 :


Hypothèse : Il existe une corrélation négative entre le nombre de jours restants avant le vol et le prix du billet.

Test à utiliser : Coefficient de corrélation de Pearson.

Variables : days_left et price.

Interprétation : Un coefficient proche de -1 indique que quand les jours diminuent, le prix augmente fortement

In [17]:

corr, p_val = pearsonr(df['days_left'], df['price'])

print(f"Coefficient de Corrélation de Pearson (R) : {corr:.4f}")
print(f"P-value : {p_val:.5f}")

Coefficient de Corrélation de Pearson (R) : -0.0919
P-value : 0.00000


"Bien que le test confirme une relation négative statistiquement certaine ($p < 0.05$), la faiblesse du coefficient $R$ (-0.09) suggère que l'impact des jours restants est secondaire par rapport à celui de la classe, ou qu'il s'exprime de manière non-linéaire (augmentation brutale au dernier moment)

In [20]:
from scipy.stats import f_oneway

# 1. Préparation des groupes (un groupe de prix par compagnie)
groupes = [df[df['airline'] == c]['price'] for c in df['airline'].unique()]

# 2. Exécution du test ANOVA
f_stat, p_val_anova = f_oneway(*groupes)

print(f"Statistique F : {f_stat:.2f}")
print(f"P-value ANOVA : {p_val_anova:.5f}")

# Interprétation
if p_val_anova < 0.05:
    print("\nRésultat Significatif : La compagnie aérienne influence fortement le prix.")
else:
    print("\nRésultat Non Significatif : Pas de différence majeure entre les compagnies.")

Statistique F : 17194.40
P-value ANOVA : 0.00000

Résultat Significatif : La compagnie aérienne influence fortement le prix.


"La variable airline possède un fort pouvoir prédictif. Le modèle ne doit pas seulement apprendre la distance ou le trajet, mais il doit impérativement ajuster sa prédiction en fonction de la signature tarifaire propre à chaque compagnie."